<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 06
## Longitudinal Analysis, Temporal Consistency, and Scenario-Alignment Gate

**Runtime:** Google Colab CPU is sufficient  
**Inputs:** completed Notebook 05 trust/QC artifacts, Notebook 04 segmentation volumes, and Notebook 01 prior reviewed FHIR Observations  
**Data boundary:** public de-identified research imaging linked only to synthetic FHIR demonstration context

### What this notebook builds

1. Reloads the three prior reviewed baseline-volume `Observation` resources.
2. Selects the current AI-derived follow-up volume for each case.
3. Calculates elapsed days, absolute volume change, and percentage volume change.
4. Applies transparent engineering categories for stable, meaningful increase, meaningful decrease, and intermediate change.
5. Withholds longitudinal interpretation when QC requires manual review.
6. Creates longitudinal tables, figures, reusable code, audit artifacts, and a scenario-alignment report.
7. Detects whether the executed outputs honestly support the planned **stable**, **progression**, and **low-confidence** demonstration stories.

### Important integrity rule

This notebook does **not** force a case to match its planned label. If the actual model-derived follow-up volume conflicts with the synthetic prior baseline, the mismatch is recorded and the competition-alignment gate remains closed.

### What this notebook intentionally does not build

- no current FHIR AI `Observation`;
- no `DiagnosticReport`, `Device`, `Provenance`, or `Task`;
- no reviewer accept/reject/correction action;
- no FHIR transaction write-back;
- no clinical interpretation or validated response criteria.

Those functions belong to Notebooks 07 and 08, after the longitudinal evidence is internally coherent.

In [1]:
# Cell 1 — Mount Drive, load project state, and enforce the Notebook 05 completion gate

from __future__ import annotations

import csv
import hashlib
import json
import math
import re
import subprocess
import sys
import textwrap
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
CONFIG_PATH = PROJECT_ROOT / "project_config.json"
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

NB05_AUDIT_PATH = PROJECT_ROOT / "evaluation/results/notebook_05_trust_robustness_audit.json"
QC_MANIFEST_PATH = PROJECT_ROOT / "data/sample_biomarkers/notebook_05/qc_case_manifest.json"
QC_SUMMARY_PATH = PROJECT_ROOT / "evaluation/results/notebook_05_trust_and_robustness/qc_case_summary.json"
SEGMENTATION_MANIFEST_PATH = PROJECT_ROOT / "data/sample_masks/notebook_04/segmentation_case_manifest.json"
DEMO_CASE_MANIFEST_PATH = PROJECT_ROOT / "data/synthetic_fhir/notebook_01/demo_case_manifest.json"
RESOURCE_INDEX_PATH = PROJECT_ROOT / "data/synthetic_fhir/notebook_01/resource_index.json"

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, allow_nan=False)
        handle.write("\n")
    temporary.replace(path)

def utc_now() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def notebook_entries(manifest: Any) -> list[dict[str, Any]]:
    if isinstance(manifest, list):
        return manifest
    if isinstance(manifest, dict):
        for key in ("notebooks", "entries", "workflow"):
            if isinstance(manifest.get(key), list):
                return manifest[key]
    raise ValueError("Unrecognized notebook_manifest.json structure")

def number_of(value: Any) -> str:
    match = re.search(r"\d+", str(value))
    return match.group(0).zfill(2) if match else str(value)

def find_entry(manifest: Any, number: str) -> dict[str, Any]:
    target = number.zfill(2)
    for entry in notebook_entries(manifest):
        candidates = [
            entry.get("number"),
            entry.get("notebook_number"),
            entry.get("id"),
            entry.get("filename"),
        ]
        if any(number_of(value) == target for value in candidates if value is not None):
            return entry
    raise KeyError(f"Notebook {target} missing from manifest")

required_paths = [
    CONFIG_PATH,
    NOTEBOOK_MANIFEST_PATH,
    NB05_AUDIT_PATH,
    QC_MANIFEST_PATH,
    QC_SUMMARY_PATH,
    SEGMENTATION_MANIFEST_PATH,
    DEMO_CASE_MANIFEST_PATH,
    RESOURCE_INDEX_PATH,
]
missing = [str(path) for path in required_paths if not path.exists() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError(
        "Notebook 05 or earlier source evidence is incomplete:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

project_config = load_json(CONFIG_PATH)
notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)
nb05_audit = load_json(NB05_AUDIT_PATH)

nb05_entry = find_entry(notebook_manifest, "05")
nb06_entry = find_entry(notebook_manifest, "06")
nb05_status = str(nb05_entry.get("status", nb05_audit.get("status", ""))).strip().lower()
if nb05_status not in {"completed", "complete", "passed"}:
    raise RuntimeError(f"Notebook 05 is not complete: {nb05_status!r}")

required_nb05_metrics = {
    "case_count": 3,
    "standard_perturbation_run_count": 12,
    "challenge_run_count": 1,
    "low_confidence_manual_review_detection_rate": 1.0,
    "preliminary_status_rate": 1.0,
    "autonomous_finalization_block_rate": 1.0,
}
nb05_metrics = nb05_audit.get("metrics", {})
failed_metrics = [
    key for key, expected in required_nb05_metrics.items()
    if float(nb05_metrics.get(key, -1)) != float(expected)
]
if failed_metrics:
    raise RuntimeError("Notebook 05 evidence gate failed: " + ", ".join(failed_metrics))

NOTEBOOK_FILENAME = nb06_entry.get(
    "filename",
    "06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb",
)
NOTEBOOK_SAVE_PATH = PROJECT_ROOT / "notebooks" / NOTEBOOK_FILENAME

LONGITUDINAL_DATA_ROOT = PROJECT_ROOT / "data/sample_biomarkers/notebook_06"
EVAL_ROOT = PROJECT_ROOT / "evaluation/results/notebook_06_longitudinal_analysis"
CASE_RESULT_ROOT = EVAL_ROOT / "case_results"
PREVIEW_ROOT = EVAL_ROOT / "previews"

LONGITUDINAL_MANIFEST_PATH = LONGITUDINAL_DATA_ROOT / "longitudinal_case_manifest.json"
LONGITUDINAL_RESULTS_JSON = EVAL_ROOT / "longitudinal_results.json"
LONGITUDINAL_RESULTS_CSV = EVAL_ROOT / "longitudinal_results.csv"
SCENARIO_ALIGNMENT_REPORT_PATH = EVAL_ROOT / "scenario_alignment_report.json"
ALIGNMENT_RECOMMENDATIONS_PATH = EVAL_ROOT / "synthetic_context_alignment_recommendations.json"
AUDIT_JSON_PATH = PROJECT_ROOT / "evaluation/results/notebook_06_longitudinal_analysis_audit.json"
AUDIT_MD_PATH = PROJECT_ROOT / "docs/NOTEBOOK_06_LONGITUDINAL_ANALYSIS.md"

for folder in (LONGITUDINAL_DATA_ROOT, EVAL_ROOT, CASE_RESULT_ROOT, PREVIEW_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 104)
print("✅ Notebook 05 completion gate passed")
print("✅ Required source manifests and prior FHIR index found")
print(f"📓 Notebook 06 path: {NOTEBOOK_SAVE_PATH}")
print("⚠️ Longitudinal thresholds are engineering display parameters, not clinical response criteria")
print("⚠️ Planned case labels will not override executed model-derived measurements")
print("=" * 104)

Mounted at /content/drive
✅ Notebook 05 completion gate passed
✅ Required source manifests and prior FHIR index found
📓 Notebook 06 path: /content/drive/MyDrive/neurofhir-qc/notebooks/06_NeuroFHIR_QC_Longitudinal_Analysis.ipynb
⚠️ Longitudinal thresholds are engineering display parameters, not clinical response criteria
⚠️ Planned case labels will not override executed model-derived measurements


In [2]:
# Cell 2 — Install the lightweight analysis runtime and record versions

packages = [
    "pandas>=2,<3",
    "matplotlib>=3.8,<4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

runtime_versions = {
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "matplotlib": matplotlib.__version__,
}

print("=" * 104)
for package_name, package_version in runtime_versions.items():
    print(f"{package_name}: {package_version}")
print("✅ CPU-only longitudinal analysis runtime is ready")
print("=" * 104)

python: 3.12.13
pandas: 2.2.2
matplotlib: 3.10.0
✅ CPU-only longitudinal analysis runtime is ready


In [3]:
# Cell 3 — Reload and validate prior reviewed Observations, segmentation volumes, and QC states

demo_manifest = load_json(DEMO_CASE_MANIFEST_PATH)
resource_index = load_json(RESOURCE_INDEX_PATH)
segmentation_manifest = load_json(SEGMENTATION_MANIFEST_PATH)
qc_manifest = load_json(QC_MANIFEST_PATH)
qc_summary = load_json(QC_SUMMARY_PATH)

CASE_ORDER = ("stable", "progression", "low-confidence")
demo_by_id = {case["case_id"]: case for case in demo_manifest.get("cases", [])}
segmentation_by_id = {case["case_id"]: case for case in segmentation_manifest.get("cases", [])}
qc_by_id = {case["case_id"]: case for case in qc_manifest.get("cases", [])}

for label, mapping in (
    ("Notebook 01 demonstration", demo_by_id),
    ("Notebook 04 segmentation", segmentation_by_id),
    ("Notebook 05 QC", qc_by_id),
):
    if set(mapping) != set(CASE_ORDER):
        raise AssertionError(f"{label} cases do not match the locked case set: {sorted(mapping)}")

index_rows = resource_index.get("resources", [])
source_contexts: dict[str, dict[str, Any]] = {}

for case_id in CASE_ORDER:
    demo_case = demo_by_id[case_id]
    segmentation_case = segmentation_by_id[case_id]
    qc_case = qc_by_id[case_id]

    observation_rows = [
        row for row in index_rows
        if row.get("case_id") == case_id and row.get("resource_type") == "Observation"
    ]
    if len(observation_rows) != 1:
        raise AssertionError(f"Expected one prior Observation for {case_id}; found {len(observation_rows)}")

    observation_path = PROJECT_ROOT / observation_rows[0]["relative_path"]
    observation = load_json(observation_path)

    expected_observation_reference = demo_case["prior_observation_reference"]
    actual_observation_reference = f"{observation.get('resourceType')}/{observation.get('id')}"
    if actual_observation_reference != expected_observation_reference:
        raise AssertionError(
            f"{case_id} prior Observation reference mismatch: "
            f"{actual_observation_reference} != {expected_observation_reference}"
        )
    if observation.get("resourceType") != "Observation" or observation.get("status") != "final":
        raise AssertionError(f"{case_id} prior measurement is not a final Observation")
    quantity = observation.get("valueQuantity", {})
    if quantity.get("system") != "http://unitsofmeasure.org" or quantity.get("code") != "mL":
        raise AssertionError(f"{case_id} prior Observation is not UCUM mL")
    baseline_volume_ml = float(quantity.get("value", 0))
    if baseline_volume_ml <= 0:
        raise AssertionError(f"{case_id} prior baseline volume is not positive")

    if observation.get("subject", {}).get("reference") != demo_case["patient_reference"]:
        raise AssertionError(f"{case_id} prior Observation subject mismatch")
    derived_references = {
        item.get("reference")
        for item in observation.get("derivedFrom", [])
        if isinstance(item, dict)
    }
    if demo_case["baseline_imaging_reference"] not in derived_references:
        raise AssertionError(f"{case_id} prior Observation does not derive from baseline ImagingStudy")

    for source_name, source_case in (
        ("segmentation", segmentation_case),
        ("QC", qc_case),
    ):
        if source_case.get("patient_reference") != demo_case["patient_reference"]:
            raise AssertionError(f"{case_id} {source_name} patient reference mismatch")
        if source_case.get("followup_imaging_reference") != demo_case["followup_imaging_reference"]:
            raise AssertionError(f"{case_id} {source_name} follow-up ImagingStudy mismatch")

    source_contexts[case_id] = {
        "demo": demo_case,
        "prior_observation": observation,
        "prior_observation_path": observation_path,
        "segmentation": segmentation_case,
        "qc": qc_case,
    }

print("=" * 104)
print("✅ Three prior reviewed FHIR Observations reloaded and validated")
print("✅ Patient, baseline ImagingStudy, follow-up ImagingStudy, segmentation, and QC references agree")
for case_id in CASE_ORDER:
    context = source_contexts[case_id]
    baseline = float(context["prior_observation"]["valueQuantity"]["value"])
    predicted = float(context["segmentation"]["region_metrics"]["whole_tumor"]["predicted_volume_ml"])
    qc_category = context["qc"]["qc"]["category"]
    print(f" - {case_id}: prior={baseline:.2f} mL | model={predicted:.2f} mL | QC={qc_category}")
print("✅ No new FHIR resource has been created")
print("=" * 104)

✅ Three prior reviewed FHIR Observations reloaded and validated
✅ Patient, baseline ImagingStudy, follow-up ImagingStudy, segmentation, and QC references agree
 - stable: prior=14.20 mL | model=19.19 mL | QC=High confidence
 - progression: prior=12.50 mL | model=20.46 mL | QC=High confidence
 - low-confidence: prior=16.10 mL | model=20.46 mL | QC=Manual review required
✅ No new FHIR resource has been created


In [4]:
# Cell 4 — Define transparent longitudinal calculations and safety-aware interpretation

LONGITUDINAL_THRESHOLDS = {
    "stable_absolute_percent_change_lt": 10.0,
    "meaningful_change_absolute_percent_change_gte": 20.0,
}
ENGINEERING_PARAMETER_NOTICE = (
    "The longitudinal display thresholds are engineering parameters for the synthetic demonstration. "
    "They are not validated clinical response criteria."
)

def parse_fhir_datetime(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00"))

def calculate_longitudinal_change(
    baseline_volume_ml: float,
    followup_volume_ml: float,
) -> dict[str, float]:
    if baseline_volume_ml <= 0:
        raise ValueError("Baseline volume must be greater than zero")
    if followup_volume_ml < 0:
        raise ValueError("Follow-up volume cannot be negative")
    absolute_change_ml = followup_volume_ml - baseline_volume_ml
    percent_change = 100.0 * absolute_change_ml / baseline_volume_ml
    return {
        "absolute_change_ml": round(absolute_change_ml, 6),
        "percent_change": round(percent_change, 6),
    }

def engineering_change_category(percent_change: float) -> str:
    stable_limit = LONGITUDINAL_THRESHOLDS["stable_absolute_percent_change_lt"]
    meaningful_limit = LONGITUDINAL_THRESHOLDS[
        "meaningful_change_absolute_percent_change_gte"
    ]
    if abs(percent_change) < stable_limit:
        return "stable"
    if percent_change >= meaningful_limit:
        return "meaningful-increase"
    if percent_change <= -meaningful_limit:
        return "meaningful-decrease"
    return "intermediate-change"

def select_current_ai_volume(
    case_id: str,
    segmentation_case: dict[str, Any],
    qc_case: dict[str, Any],
) -> dict[str, Any]:
    baseline_model_volume = float(
        segmentation_case["region_metrics"]["whole_tumor"]["predicted_volume_ml"]
    )

    if case_id == "low-confidence":
        challenge = qc_case.get("low_confidence_demo_challenge")
        if not isinstance(challenge, dict):
            raise AssertionError("Low-confidence case is missing its severe synthetic challenge")
        return {
            "selected_current_ai_volume_ml": float(challenge["perturbed_volume_ml"]),
            "selected_current_volume_source": "Notebook 05 severe synthetic challenge output",
            "selected_current_volume_is_synthetic_challenge": True,
            "baseline_model_followup_volume_ml": baseline_model_volume,
            "selected_mask_file": challenge["output_mask_file"],
        }

    return {
        "selected_current_ai_volume_ml": baseline_model_volume,
        "selected_current_volume_source": "Notebook 04 baseline model output",
        "selected_current_volume_is_synthetic_challenge": False,
        "baseline_model_followup_volume_ml": baseline_model_volume,
        "selected_mask_file": segmentation_case["outputs"]["predicted_whole_tumor_mask_file"],
    }

def expected_alignment(case_id: str, raw_category: str, qc_category: str) -> tuple[bool, str]:
    if case_id == "stable":
        passed = raw_category == "stable" and qc_category == "High confidence"
        reason = (
            "Expected a stable volume change with High confidence."
            if passed
            else "Executed model-derived change does not support the planned stable story."
        )
        return passed, reason

    if case_id == "progression":
        passed = raw_category == "meaningful-increase" and qc_category == "High confidence"
        reason = (
            "Meaningful increase and High confidence support the progression story."
            if passed
            else "Executed output does not support the planned progression story."
        )
        return passed, reason

    passed = qc_category == "Manual review required"
    reason = (
        "Manual-review triage supports the low-confidence safety story."
        if passed
        else "Low-confidence case was not routed to manual review."
    )
    return passed, reason

def recommended_synthetic_baseline_for_planned_change(
    current_volume_ml: float,
    planned_percent_change: float,
) -> float:
    denominator = 1.0 + planned_percent_change / 100.0
    if denominator <= 0:
        raise ValueError("Planned percent change produces a non-positive denominator")
    return round(current_volume_ml / denominator, 6)

print("✅ Longitudinal calculation functions defined")
print(f"✅ Stable display gate: |change| < {LONGITUDINAL_THRESHOLDS['stable_absolute_percent_change_lt']:.1f}%")
print(
    "✅ Meaningful change gate: |change| ≥ "
    f"{LONGITUDINAL_THRESHOLDS['meaningful_change_absolute_percent_change_gte']:.1f}%"
)
print("⚠️ These gates are engineering display parameters, not clinical response criteria")

✅ Longitudinal calculation functions defined
✅ Stable display gate: |change| < 10.0%
✅ Meaningful change gate: |change| ≥ 20.0%
⚠️ These gates are engineering display parameters, not clinical response criteria


In [5]:
# Cell 5 — Calculate longitudinal change, apply QC-aware interpretation, and persist case evidence

longitudinal_cases: list[dict[str, Any]] = []
summary_rows: list[dict[str, Any]] = []

for case_id in CASE_ORDER:
    context = source_contexts[case_id]
    demo_case = context["demo"]
    prior_observation = context["prior_observation"]
    segmentation_case = context["segmentation"]
    qc_case = context["qc"]

    baseline_datetime = parse_fhir_datetime(demo_case["baseline_datetime"])
    followup_datetime = parse_fhir_datetime(demo_case["followup_datetime"])
    interval_days = (followup_datetime - baseline_datetime).total_seconds() / 86400.0
    if interval_days <= 0:
        raise AssertionError(f"{case_id} follow-up does not occur after baseline")

    baseline_volume_ml = float(prior_observation["valueQuantity"]["value"])
    current_selection = select_current_ai_volume(case_id, segmentation_case, qc_case)
    current_volume_ml = float(current_selection["selected_current_ai_volume_ml"])
    change = calculate_longitudinal_change(baseline_volume_ml, current_volume_ml)
    raw_category = engineering_change_category(change["percent_change"])

    qc_category = str(qc_case["qc"]["category"])
    qc_score = float(qc_case["qc"]["qc_score"])
    manual_review_required = qc_category == "Manual review required"

    if manual_review_required:
        longitudinal_interpretation = "withheld-pending-human-review"
        display_label = "Longitudinal interpretation withheld — manual review required"
    else:
        longitudinal_interpretation = raw_category
        display_label = {
            "stable": "Stable by engineering display threshold",
            "meaningful-increase": "Meaningful volume increase by engineering display threshold",
            "meaningful-decrease": "Meaningful volume decrease by engineering display threshold",
            "intermediate-change": "Intermediate volume change — review recommended",
        }[raw_category]

    alignment_passed, alignment_reason = expected_alignment(
        case_id,
        raw_category,
        qc_category,
    )

    result = {
        "case_id": case_id,
        "source_case_id": segmentation_case["source_case_id"],
        "patient_reference": demo_case["patient_reference"],
        "condition_reference": demo_case["condition_reference"],
        "baseline_imaging_reference": demo_case["baseline_imaging_reference"],
        "followup_imaging_reference": demo_case["followup_imaging_reference"],
        "prior_observation_reference": demo_case["prior_observation_reference"],
        "baseline_datetime": demo_case["baseline_datetime"],
        "followup_datetime": demo_case["followup_datetime"],
        "interval_days": round(interval_days, 3),
        "prior_reviewed_baseline_volume_ml": round(baseline_volume_ml, 6),
        **current_selection,
        **change,
        "raw_engineering_change_category": raw_category,
        "qc_score": round(qc_score, 6),
        "qc_category": qc_category,
        "longitudinal_interpretation": longitudinal_interpretation,
        "longitudinal_display_label": display_label,
        "planned_followup_reference_volume_ml": demo_case[
            "planned_followup_reference_volume_ml"
        ],
        "planned_percent_change": demo_case["planned_percent_change"],
        "planned_change_category": demo_case["planned_change_category"],
        "scenario_alignment_passed": alignment_passed,
        "scenario_alignment_reason": alignment_reason,
        "workflow_state": {
            "ai_result_status": "preliminary",
            "human_review_required": True,
            "manual_review_required": manual_review_required,
            "autonomous_finalization_allowed": False,
            "longitudinal_interpretation_withheld": manual_review_required,
        },
        "source_artifacts": {
            "prior_observation_file": context["prior_observation_path"]
            .relative_to(PROJECT_ROOT)
            .as_posix(),
            "segmentation_case_result_file": segmentation_case["case_result_file"],
            "qc_case_result_file": qc_case["case_result_file"],
            "selected_mask_file": current_selection["selected_mask_file"],
        },
        "interpretation_boundary": (
            "Synthetic longitudinal workflow evidence only. The public image is not claimed "
            "to be a real follow-up scan from the synthetic patient."
        ),
    }

    case_result_path = CASE_RESULT_ROOT / f"{case_id}_longitudinal_result.json"
    write_json(case_result_path, result)
    result["case_result_file"] = case_result_path.relative_to(PROJECT_ROOT).as_posix()
    longitudinal_cases.append(result)

    summary_rows.append({
        "case_id": case_id,
        "source_case_id": result["source_case_id"],
        "baseline_volume_ml": result["prior_reviewed_baseline_volume_ml"],
        "current_ai_volume_ml": result["selected_current_ai_volume_ml"],
        "absolute_change_ml": result["absolute_change_ml"],
        "percent_change": result["percent_change"],
        "raw_change_category": result["raw_engineering_change_category"],
        "qc_score": result["qc_score"],
        "qc_category": result["qc_category"],
        "longitudinal_interpretation": result["longitudinal_interpretation"],
        "scenario_alignment_passed": result["scenario_alignment_passed"],
        "observation_status": "preliminary",
        "autonomous_finalization_allowed": False,
    })

if len(longitudinal_cases) != 3:
    raise AssertionError("Expected exactly three longitudinal cases")
if any(case["workflow_state"]["autonomous_finalization_allowed"] for case in longitudinal_cases):
    raise AssertionError("Autonomous finalization was incorrectly allowed")
if any(case["workflow_state"]["ai_result_status"] != "preliminary" for case in longitudinal_cases):
    raise AssertionError("Every AI result must remain preliminary")
low_case = next(case for case in longitudinal_cases if case["case_id"] == "low-confidence")
if low_case["longitudinal_interpretation"] != "withheld-pending-human-review":
    raise AssertionError("Low-confidence longitudinal interpretation was not withheld")

summary_dataframe = pd.DataFrame(summary_rows)
summary_dataframe.to_csv(LONGITUDINAL_RESULTS_CSV, index=False)

write_json(LONGITUDINAL_RESULTS_JSON, {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "generated_utc": utc_now(),
    "notebook_number": "06",
    "engineering_parameter_notice": ENGINEERING_PARAMETER_NOTICE,
    "thresholds": LONGITUDINAL_THRESHOLDS,
    "case_count": len(longitudinal_cases),
    "rows": summary_rows,
})

write_json(LONGITUDINAL_MANIFEST_PATH, {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "generated_utc": utc_now(),
    "notebook_number": "06",
    "data_governance": {
        "public_deidentified_imaging_only": True,
        "synthetic_fhir_context_only": True,
        "real_longitudinal_patient_linkage_claimed": False,
        "clinical_response_criteria_claimed": False,
    },
    "safety": {
        "all_ai_outputs_preliminary": True,
        "human_review_required_before_final": True,
        "autonomous_finalization_allowed": False,
        "manual_review_suppresses_longitudinal_interpretation": True,
    },
    "cases": longitudinal_cases,
})

print("=" * 104)
print("✅ Longitudinal calculations completed for 3/3 cases")
for row in summary_rows:
    print(
        f" - {row['case_id']}: {row['baseline_volume_ml']:.2f} → "
        f"{row['current_ai_volume_ml']:.2f} mL | "
        f"{row['percent_change']:+.1f}% | "
        f"{row['longitudinal_interpretation']} | "
        f"alignment={row['scenario_alignment_passed']}"
    )
print("✅ Low-confidence longitudinal interpretation withheld pending human review")
print("✅ All results remain preliminary; autonomous finalization blocked")
print("=" * 104)

✅ Longitudinal calculations completed for 3/3 cases
 - stable: 14.20 → 19.19 mL | +35.1% | meaningful-increase | alignment=False
 - progression: 12.50 → 20.46 mL | +63.7% | meaningful-increase | alignment=True
 - low-confidence: 16.10 → 1.63 mL | -89.9% | withheld-pending-human-review | alignment=True
✅ Low-confidence longitudinal interpretation withheld pending human review
✅ All results remain preliminary; autonomous finalization blocked


In [6]:
# Cell 6 — Create visual evidence, scenario-alignment report, and non-applied repair recommendations

summary_dataframe = pd.DataFrame(summary_rows).set_index("case_id").loc[list(CASE_ORDER)].reset_index()

baseline_current_preview = PREVIEW_ROOT / "baseline_vs_current_volume.png"
figure, axis = plt.subplots(figsize=(9, 5))
x_positions = list(range(len(summary_dataframe)))
bar_width = 0.36
axis.bar(
    [x - bar_width / 2 for x in x_positions],
    summary_dataframe["baseline_volume_ml"],
    width=bar_width,
    label="Prior reviewed baseline",
)
axis.bar(
    [x + bar_width / 2 for x in x_positions],
    summary_dataframe["current_ai_volume_ml"],
    width=bar_width,
    label="Selected current AI output",
)
axis.set_xticks(x_positions)
axis.set_xticklabels(summary_dataframe["case_id"])
axis.set_ylabel("Volume (mL)")
axis.set_title("NeuroFHIR-QC longitudinal volume comparison")
axis.legend()
figure.tight_layout()
figure.savefig(baseline_current_preview, dpi=180, bbox_inches="tight")
plt.close(figure)

percent_change_preview = PREVIEW_ROOT / "longitudinal_percent_change.png"
figure, axis = plt.subplots(figsize=(9, 5))
axis.bar(summary_dataframe["case_id"], summary_dataframe["percent_change"])
axis.axhline(
    LONGITUDINAL_THRESHOLDS["meaningful_change_absolute_percent_change_gte"],
    linestyle="--",
    label="Meaningful increase gate",
)
axis.axhline(
    -LONGITUDINAL_THRESHOLDS["meaningful_change_absolute_percent_change_gte"],
    linestyle="--",
    label="Meaningful decrease gate",
)
axis.axhline(
    LONGITUDINAL_THRESHOLDS["stable_absolute_percent_change_lt"],
    linestyle=":",
    label="Stable band",
)
axis.axhline(
    -LONGITUDINAL_THRESHOLDS["stable_absolute_percent_change_lt"],
    linestyle=":",
)
axis.set_ylabel("Change from prior reviewed volume (%)")
axis.set_title("Longitudinal change with engineering display gates")
axis.legend()
figure.tight_layout()
figure.savefig(percent_change_preview, dpi=180, bbox_inches="tight")
plt.close(figure)

alignment_rows = []
recommendations = []

for case in longitudinal_cases:
    alignment_rows.append({
        "case_id": case["case_id"],
        "planned_change_category": case["planned_change_category"],
        "executed_raw_change_category": case["raw_engineering_change_category"],
        "qc_category": case["qc_category"],
        "longitudinal_interpretation": case["longitudinal_interpretation"],
        "alignment_passed": case["scenario_alignment_passed"],
        "reason": case["scenario_alignment_reason"],
    })

    if not case["scenario_alignment_passed"] and case["case_id"] != "low-confidence":
        recommended_baseline = recommended_synthetic_baseline_for_planned_change(
            case["selected_current_ai_volume_ml"],
            float(case["planned_percent_change"]),
        )
        recommendations.append({
            "case_id": case["case_id"],
            "recommendation_status": "not_applied",
            "current_prior_reviewed_baseline_volume_ml": case[
                "prior_reviewed_baseline_volume_ml"
            ],
            "executed_current_ai_volume_ml": case["selected_current_ai_volume_ml"],
            "planned_percent_change": case["planned_percent_change"],
            "mathematically_aligned_synthetic_baseline_volume_ml": recommended_baseline,
            "required_action": (
                "Regenerate the synthetic prior Observation and dependent audits, reseed the "
                "FHIR source context, and rerun Notebook 06 before Notebook 07."
            ),
            "integrity_notice": (
                "This is a transparent synthetic-context design recommendation, not a clinical "
                "correction and not an automatically applied data mutation."
            ),
        })

alignment_pass_count = sum(bool(row["alignment_passed"]) for row in alignment_rows)
alignment_pass_rate = alignment_pass_count / len(alignment_rows)
competition_alignment_ready = alignment_pass_count == len(alignment_rows)

alignment_report = {
    "project_name": project_config["project_name"],
    "generated_utc": utc_now(),
    "case_count": len(alignment_rows),
    "alignment_pass_count": alignment_pass_count,
    "alignment_pass_rate": round(alignment_pass_rate, 6),
    "competition_alignment_ready": competition_alignment_ready,
    "rows": alignment_rows,
    "gate_rule": (
        "Notebook 07 should begin only after all three executed cases support their planned "
        "competition behavior without relabeling or substituting planned values for model outputs."
    ),
}
write_json(SCENARIO_ALIGNMENT_REPORT_PATH, alignment_report)

write_json(ALIGNMENT_RECOMMENDATIONS_PATH, {
    "project_name": project_config["project_name"],
    "generated_utc": utc_now(),
    "recommendations_are_applied": False,
    "recommendation_count": len(recommendations),
    "recommendations": recommendations,
})

alignment_preview = PREVIEW_ROOT / "scenario_alignment_summary.png"
alignment_dataframe = pd.DataFrame(alignment_rows)
figure, axis = plt.subplots(figsize=(9, 4.5))
axis.bar(
    alignment_dataframe["case_id"],
    alignment_dataframe["alignment_passed"].astype(int),
)
axis.set_ylim(0, 1.15)
axis.set_yticks([0, 1])
axis.set_yticklabels(["Mismatch", "Aligned"])
axis.set_title("Executed case behavior versus planned competition story")
figure.tight_layout()
figure.savefig(alignment_preview, dpi=180, bbox_inches="tight")
plt.close(figure)

for path in (
    baseline_current_preview,
    percent_change_preview,
    alignment_preview,
    SCENARIO_ALIGNMENT_REPORT_PATH,
    ALIGNMENT_RECOMMENDATIONS_PATH,
):
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(path)

print("=" * 104)
print(f"✅ Longitudinal visual evidence created: {len(list(PREVIEW_ROOT.glob('*.png')))} figures")
print(
    f"📋 Scenario alignment: {alignment_pass_count}/{len(alignment_rows)} "
    f"({alignment_pass_rate:.1%})"
)
if competition_alignment_ready:
    print("✅ Competition alignment gate passed")
else:
    print("⚠️ Competition alignment gate is NOT ready")
    for row in alignment_rows:
        if not row["alignment_passed"]:
            print(f"   - {row['case_id']}: {row['reason']}")
    print("⚠️ No synthetic source value was silently changed")
    print(f"📄 Non-applied recommendations: {ALIGNMENT_RECOMMENDATIONS_PATH}")
print("=" * 104)

✅ Longitudinal visual evidence created: 3 figures
📋 Scenario alignment: 2/3 (66.7%)
⚠️ Competition alignment gate is NOT ready
   - stable: Executed model-derived change does not support the planned stable story.
⚠️ No synthetic source value was silently changed
📄 Non-applied recommendations: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_06_longitudinal_analysis/synthetic_context_alignment_recommendations.json


In [7]:
# Cell 7 — Create reusable longitudinal code, verification script, requirements, and documentation

LONGITUDINAL_ENGINE_PATH = PROJECT_ROOT / "backend/app/services/longitudinal_engine.py"
VERIFY_SCRIPT_PATH = PROJECT_ROOT / "scripts/verify_longitudinal_results.py"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements/longitudinal.txt"
TECH_DOC_PATH = PROJECT_ROOT / "docs/LONGITUDINAL_ANALYSIS.md"

longitudinal_engine_source = '"""Reusable longitudinal helpers for NeuroFHIR-QC.\n\nResearch demonstration only. Thresholds are engineering display parameters, not\nvalidated clinical response criteria.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom datetime import datetime\n\n\n@dataclass(frozen=True)\nclass LongitudinalThresholds:\n    stable_absolute_percent_change_lt: float = 10.0\n    meaningful_change_absolute_percent_change_gte: float = 20.0\n\n\ndef parse_fhir_datetime(value: str) -> datetime:\n    return datetime.fromisoformat(value.replace("Z", "+00:00"))\n\n\ndef calculate_change(baseline_volume_ml: float, followup_volume_ml: float) -> dict[str, float]:\n    if baseline_volume_ml <= 0:\n        raise ValueError("Baseline volume must be greater than zero")\n    if followup_volume_ml < 0:\n        raise ValueError("Follow-up volume cannot be negative")\n    absolute_change_ml = followup_volume_ml - baseline_volume_ml\n    percent_change = 100.0 * absolute_change_ml / baseline_volume_ml\n    return {\n        "absolute_change_ml": round(absolute_change_ml, 6),\n        "percent_change": round(percent_change, 6),\n    }\n\n\ndef categorize_change(\n    percent_change: float,\n    thresholds: LongitudinalThresholds = LongitudinalThresholds(),\n) -> str:\n    if abs(percent_change) < thresholds.stable_absolute_percent_change_lt:\n        return "stable"\n    if percent_change >= thresholds.meaningful_change_absolute_percent_change_gte:\n        return "meaningful-increase"\n    if percent_change <= -thresholds.meaningful_change_absolute_percent_change_gte:\n        return "meaningful-decrease"\n    return "intermediate-change"\n\n\ndef interpretation_state(raw_category: str, qc_category: str) -> str:\n    if qc_category == "Manual review required":\n        return "withheld-pending-human-review"\n    return raw_category\n'
verify_script_source = '"""Verify persisted Notebook 06 evidence without recalculating model inference."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nMANIFEST = PROJECT_ROOT / "data/sample_biomarkers/notebook_06/longitudinal_case_manifest.json"\nRESULTS = PROJECT_ROOT / "evaluation/results/notebook_06_longitudinal_analysis/longitudinal_results.json"\nALIGNMENT = PROJECT_ROOT / "evaluation/results/notebook_06_longitudinal_analysis/scenario_alignment_report.json"\n\n\ndef load(path: Path):\n    with path.open("r", encoding="utf-8") as handle:\n        return json.load(handle)\n\n\ndef main() -> None:\n    manifest = load(MANIFEST)\n    results = load(RESULTS)\n    alignment = load(ALIGNMENT)\n\n    cases = manifest.get("cases", [])\n    if len(cases) != 3 or int(results.get("case_count", 0)) != 3:\n        raise SystemExit("Expected exactly three longitudinal cases")\n\n    if any(case["workflow_state"]["ai_result_status"] != "preliminary" for case in cases):\n        raise SystemExit("Every AI result must remain preliminary")\n    if any(case["workflow_state"]["autonomous_finalization_allowed"] for case in cases):\n        raise SystemExit("Autonomous finalization must remain blocked")\n\n    low = next(case for case in cases if case["case_id"] == "low-confidence")\n    if low["qc_category"] != "Manual review required":\n        raise SystemExit("Low-confidence case is not in manual review")\n    if low["longitudinal_interpretation"] != "withheld-pending-human-review":\n        raise SystemExit("Low-confidence interpretation was not withheld")\n\n    if int(alignment.get("case_count", 0)) != 3:\n        raise SystemExit("Scenario-alignment report is incomplete")\n\n    print("Notebook 06 longitudinal evidence passed structural and safety verification.")\n    if not alignment.get("competition_alignment_ready", False):\n        print("WARNING: competition scenario alignment remains unresolved.")\n\n\nif __name__ == "__main__":\n    main()\n'

technical_documentation = f"""# NeuroFHIR-QC Longitudinal Analysis

Generated by `{NOTEBOOK_FILENAME}`.

## Inputs

- Three synthetic prior reviewed baseline-volume FHIR Observations.
- Three Notebook 04 model-derived segmentation volumes.
- Three Notebook 05 QC states.
- One severe synthetic challenge output for the low-confidence demonstration.

## Calculations

- elapsed days between baseline and follow-up;
- absolute volume change in mL;
- percentage volume change;
- transparent engineering display category;
- QC-aware suppression of longitudinal interpretation;
- planned-versus-executed scenario-alignment gate.

## Safety

- Every current AI result remains preliminary.
- Human review is required before final status.
- Low-confidence longitudinal interpretation is withheld.
- No current FHIR AI Observation or transaction write-back is created.
- Thresholds are not represented as validated clinical response criteria.

## Alignment integrity

The notebook never replaces an executed model value with a planned scenario value. If a
synthetic prior baseline does not support the planned demonstration story, the mismatch is
reported in `{SCENARIO_ALIGNMENT_REPORT_PATH.relative_to(PROJECT_ROOT).as_posix()}`.
Any mathematically aligned synthetic-context recommendation is written separately and is
explicitly marked as not applied.
"""

for path in (
    LONGITUDINAL_ENGINE_PATH,
    VERIFY_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    TECH_DOC_PATH,
):
    path.parent.mkdir(parents=True, exist_ok=True)

LONGITUDINAL_ENGINE_PATH.write_text(
    textwrap.dedent(longitudinal_engine_source).strip() + "\n",
    encoding="utf-8",
)
VERIFY_SCRIPT_PATH.write_text(
    textwrap.dedent(verify_script_source).strip() + "\n",
    encoding="utf-8",
)
REQUIREMENTS_PATH.write_text(
    "\n".join([
        f"pandas=={runtime_versions['pandas']}",
        f"matplotlib=={runtime_versions['matplotlib']}",
    ]) + "\n",
    encoding="utf-8",
)
TECH_DOC_PATH.write_text(
    textwrap.dedent(technical_documentation).strip() + "\n",
    encoding="utf-8",
)

for path in (LONGITUDINAL_ENGINE_PATH, VERIFY_SCRIPT_PATH):
    compile(path.read_text(encoding="utf-8"), str(path), "exec")

subprocess.check_call([sys.executable, str(VERIFY_SCRIPT_PATH)])

print("✅ Reusable longitudinal engine, verification script, requirements, and documentation created")

✅ Reusable longitudinal engine, verification script, requirements, and documentation created


In [8]:
# Cell 8 — Final audit, checksums, and notebook-manifest update

longitudinal_manifest = load_json(LONGITUDINAL_MANIFEST_PATH)
longitudinal_results = load_json(LONGITUDINAL_RESULTS_JSON)
alignment_report = load_json(SCENARIO_ALIGNMENT_REPORT_PATH)
alignment_recommendations = load_json(ALIGNMENT_RECOMMENDATIONS_PATH)

cases = longitudinal_manifest.get("cases", [])
if len(cases) != 3:
    raise AssertionError("Expected three longitudinal cases")
if int(longitudinal_results.get("case_count", 0)) != 3:
    raise AssertionError("Longitudinal results case count is incomplete")
if any(case["workflow_state"]["ai_result_status"] != "preliminary" for case in cases):
    raise AssertionError("Preliminary-status retention failed")
if any(case["workflow_state"]["autonomous_finalization_allowed"] for case in cases):
    raise AssertionError("Autonomous-finalization blocking failed")

low_case = next(case for case in cases if case["case_id"] == "low-confidence")
if low_case["qc_category"] != "Manual review required":
    raise AssertionError("Low-confidence QC category is incorrect")
if not low_case["workflow_state"]["longitudinal_interpretation_withheld"]:
    raise AssertionError("Low-confidence longitudinal interpretation was not withheld")

core_files = [
    LONGITUDINAL_MANIFEST_PATH,
    LONGITUDINAL_RESULTS_JSON,
    LONGITUDINAL_RESULTS_CSV,
    SCENARIO_ALIGNMENT_REPORT_PATH,
    ALIGNMENT_RECOMMENDATIONS_PATH,
    LONGITUDINAL_ENGINE_PATH,
    VERIFY_SCRIPT_PATH,
    REQUIREMENTS_PATH,
    TECH_DOC_PATH,
]
core_files += sorted(CASE_RESULT_ROOT.glob("*.json"))
core_files += sorted(PREVIEW_ROOT.glob("*.png"))

missing_outputs = [
    str(path) for path in core_files
    if not path.exists() or path.stat().st_size == 0
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing Notebook 06 evidence:\n"
        + "\n".join(f" - {path}" for path in missing_outputs)
    )

checksums = []
for path in sorted({path.resolve() for path in core_files}, key=str):
    checksums.append({
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

source_observation_validation_rate = sum(
    case["prior_reviewed_baseline_volume_ml"] > 0 for case in cases
) / len(cases)
calculation_success_rate = sum(
    math.isfinite(float(case["percent_change"])) for case in cases
) / len(cases)
preliminary_status_rate = sum(
    case["workflow_state"]["ai_result_status"] == "preliminary" for case in cases
) / len(cases)
autonomous_finalization_block_rate = sum(
    not case["workflow_state"]["autonomous_finalization_allowed"] for case in cases
) / len(cases)
low_confidence_interpretation_withholding_rate = float(
    low_case["longitudinal_interpretation"] == "withheld-pending-human-review"
)

competition_alignment_ready = bool(
    alignment_report.get("competition_alignment_ready", False)
)
next_step = (
    "07 — FHIR Evidence and Write-back"
    if competition_alignment_ready
    else "Resolve synthetic-context scenario alignment, then rerun Notebook 06 before Notebook 07"
)

final_audit = {
    "project_name": project_config["project_name"],
    "project_version": project_config.get("version", "0.1.0"),
    "notebook_number": "06",
    "notebook_filename": NOTEBOOK_FILENAME,
    "status": "completed",
    "audited_utc": utc_now(),
    "notebook_saved_in_drive": (
        NOTEBOOK_SAVE_PATH.exists() and NOTEBOOK_SAVE_PATH.stat().st_size > 0
    ),
    "metrics": {
        "case_count": 3,
        "source_prior_observation_validation_rate": round(
            source_observation_validation_rate, 6
        ),
        "longitudinal_calculation_success_rate": round(
            calculation_success_rate, 6
        ),
        "preliminary_status_rate": round(preliminary_status_rate, 6),
        "autonomous_finalization_block_rate": round(
            autonomous_finalization_block_rate, 6
        ),
        "low_confidence_interpretation_withholding_rate": round(
            low_confidence_interpretation_withholding_rate, 6
        ),
        "scenario_alignment_pass_rate": alignment_report[
            "alignment_pass_rate"
        ],
        "competition_alignment_ready": competition_alignment_ready,
        "preview_count": len(list(PREVIEW_ROOT.glob("*.png"))),
    },
    "scope": {
        "prior_reviewed_observations_loaded": True,
        "absolute_change_calculated": True,
        "percentage_change_calculated": True,
        "temporal_consistency_checked": True,
        "scenario_alignment_checked": True,
        "current_ai_observation_created": False,
        "diagnostic_report_created": False,
        "human_review_transition_executed": False,
        "ai_result_writeback_performed": False,
    },
    "safety": {
        "all_ai_outputs_preliminary": True,
        "human_review_required_before_final": True,
        "low_confidence_interpretation_withheld": True,
        "planned_values_not_substituted_for_executed_outputs": True,
        "synthetic_alignment_recommendations_applied": False,
        "thresholds_are_engineering_parameters": True,
        "clinical_validation_claimed": False,
    },
    "scenario_alignment": alignment_report,
    "alignment_recommendations": alignment_recommendations,
    "checksum_inventory": checksums,
    "next_step": next_step,
}
write_json(AUDIT_JSON_PATH, final_audit)

outcome_lines = "\n".join(
    f"- {case['case_id']}: {case['percent_change']:+.1f}% | "
    f"{case['longitudinal_interpretation']} | "
    f"alignment={case['scenario_alignment_passed']}"
    for case in cases
)
AUDIT_MD_PATH.write_text(
    textwrap.dedent(f"""
    # Notebook 06 — Longitudinal Analysis

    **Status:** completed
    **Audited:** {final_audit['audited_utc']}

    - Three prior reviewed FHIR Observations were validated.
    - Absolute and percentage volume change were calculated for 3/3 cases.
    - All current AI outputs remain preliminary.
    - Autonomous finalization remains blocked for 3/3 cases.
    - Low-confidence longitudinal interpretation was withheld.
    - Scenario alignment passed for {alignment_report['alignment_pass_count']}/3 cases.
    - Competition alignment ready: {competition_alignment_ready}.

    ## Outcomes

    {outcome_lines}

    The thresholds are engineering display parameters, not clinical response criteria.
    Notebook 06 did not create current FHIR AI evidence, execute human review, or perform write-back.

    ## Next step

    {next_step}
    """).strip() + "\n",
    encoding="utf-8",
)

nb06_entry["status"] = "completed"
nb06_entry["completed_utc"] = final_audit["audited_utc"]
nb06_entry["longitudinal_calculation_success_rate"] = round(
    calculation_success_rate, 6
)
nb06_entry["low_confidence_interpretation_withholding_rate"] = round(
    low_confidence_interpretation_withholding_rate, 6
)
nb06_entry["scenario_alignment_pass_rate"] = alignment_report[
    "alignment_pass_rate"
]
nb06_entry["competition_alignment_ready"] = competition_alignment_ready
nb06_entry["audit_path"] = AUDIT_JSON_PATH.relative_to(PROJECT_ROOT).as_posix()
write_json(NOTEBOOK_MANIFEST_PATH, notebook_manifest)

print("=" * 104)
print("✅ Notebook 06 longitudinal evidence passed structural and safety checks")
print("✅ Three prior reviewed Observations validated")
print("✅ Longitudinal calculations completed for 3/3 cases")
print("✅ Low-confidence interpretation withheld; finalization blocked 3/3")
print(
    f"📋 Scenario alignment: {alignment_report['alignment_pass_count']}/3 "
    f"({alignment_report['alignment_pass_rate']:.1%})"
)
print(f"✅ Audit JSON: {AUDIT_JSON_PATH}")
print("📓 Manifest status: completed")
if competition_alignment_ready:
    print("➡️ Notebook 07 — FHIR Evidence and Write-back may begin")
else:
    print("⛔ Do not begin Notebook 07 yet")
    print("➡️ Resolve the synthetic-context alignment warning and rerun Notebook 06")
print("=" * 104)

✅ Notebook 06 longitudinal evidence passed structural and safety checks
✅ Three prior reviewed Observations validated
✅ Longitudinal calculations completed for 3/3 cases
✅ Low-confidence interpretation withheld; finalization blocked 3/3
📋 Scenario alignment: 2/3 (66.7%)
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_06_longitudinal_analysis_audit.json
📓 Manifest status: completed
⛔ Do not begin Notebook 07 yet
➡️ Resolve the synthetic-context alignment warning and rerun Notebook 06
